# Numerikus módszerek – 10. hét előadás
## Interpoláció: Lagrange-alak · Hibaformula · Newton-alak · Hermite · Csebisev

**Tartalom:**
1. Az interpoláció alapfeladata
2. Lagrange-alak
3. Hibaformula
4. Newton-alak (osztott differenciák)
5. Hermite-interpoláció
6. Csebisev-polinomok és optimális alappontok
7. Inverz interpoláció

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from numpy.polynomial import polynomial as P

np.set_printoptions(precision=8, suppress=True)
plt.rcParams['figure.figsize'] = (9, 4)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.4

---
## 1. Az interpoláció alapfeladata

**Motiváció:** A gyakorlatban sokszor egy „drágán" számolható függvény helyett egyszerűbbet (általában polinomot) használunk.
- **Interpolációs feladat:** adott pontokra pontosan illeszkedő polinomot keresünk.
- **Approximációs feladat:** mérési adatokhoz a legközelebbi polinomot keresünk (legkisebb négyzetek).

> **Definíció.**  
> Adottak az $x_0, x_1, \ldots, x_n \in [a;b]$ különböző alappontok és az $y_0, y_1, \ldots, y_n \in \mathbb{R}$ függvényértékek. Olyan $p_n \in P_n$ polinomot keresünk, melyre
> $$p_n(x_i) = y_i, \quad (i = 0, 1, \ldots, n).$$
> Ezt *interpolációs polinomnak* nevezzük.

> **Tétel (létezés és egyértelműség).**  
> $\exists!\, p_n \in P_n : p_n(x_i) = y_i\; (i = 0, \ldots, n).$

**Biz. vázlat:** Vandermonde-mátrixos LER – létezés és egyértelműség.  
**Megjegyzés:** A Vandermonde-mátrix rosszul kondicionált, ezért a hatvány-alak helyett Lagrange- vagy Newton-bázist használunk.

**Alkalmazások:** numerikus integrálás, diff.egyenlet módszerek, grafika, képfeldolgozás, meteorológia, GIS.

---
## 2. Lagrange-alak

> **Definíció (Lagrange-alappolinomok).**
> $$\ell_k(x) = \prod_{\substack{j=0 \\ j\ne k}}^{n} \frac{x - x_j}{x_k - x_j}, \quad k = 0, 1, \ldots, n.$$

> **Tétel (tulajdonságok).**
> 1. $\ell_k(x_i) = \delta_{ki}$ (Kronecker-delta)
> 2. $\ell_k(x) = \dfrac{\omega_n(x)}{(x-x_k)\,\omega_n'(x_k)}$, ahol $\omega_n(x) = \prod_{j=0}^{n}(x-x_j)$
> 3. Az interpolációs polinom: $L_n(x) := \sum_{k=0}^{n} y_k \ell_k(x) \equiv p_n(x)$

In [ ]:
def lagrange_basis(xs, k, t):
    """k-adik Lagrange-alappolinom értéke t-ben."""
    result = 1.0
    for j, xj in enumerate(xs):
        if j != k:
            result *= (t - xj) / (xs[k] - xj)
    return result

def lagrange_interp(xs, ys, t):
    """Lagrange-interpoláció értéke t-ben."""
    return sum(ys[k] * lagrange_basis(xs, k, t) for k in range(len(xs)))

# Szemléltetés: sin(x) közelítése 4 egyenletes alapponttal [-π/2, π/2]-n
xs = np.linspace(-np.pi/2, np.pi/2, 4)
ys = np.sin(xs)

t = np.linspace(-np.pi/2 - 0.3, np.pi/2 + 0.3, 400)
p_vals = np.array([lagrange_interp(xs, ys, ti) for ti in t])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Bal: az alappolinomok
colors = ['tab:blue', 'tab:orange', 'tab:green', 'tab:red']
for k in range(len(xs)):
    lk = np.array([lagrange_basis(xs, k, ti) for ti in t])
    ax1.plot(t, lk, color=colors[k], label=f'$\\ell_{k}(x)$')
ax1.scatter(xs, np.ones_like(xs)*0, s=60, color='k', zorder=5)
ax1.axhline(0, color='k', lw=0.7); ax1.axhline(1, color='k', lw=0.7, ls=':')
ax1.set_ylim(-0.6, 1.3)
ax1.set_title('Lagrange-alappolinomok ($n=3$)')
ax1.legend(fontsize=9)

# Jobb: interpoláló polinom
ax2.plot(t, np.sin(t), 'b-', lw=2, label='$\\sin(x)$')
ax2.plot(t, p_vals, 'r--', lw=1.5, label='$L_3(x)$')
ax2.scatter(xs, ys, s=70, color='k', zorder=5, label='alappontok')
ax2.set_title('Lagrange-interpoláció: $\\sin(x)$, $n=3$')
ax2.legend()

plt.tight_layout()
plt.show()

---
## 3. Hibaformula

> **Tétel (hibaformula).**  
> Legyen $x \in \mathbb{R}$ tetszőleges, $[a;b]$ az $x_0, \ldots, x_n$ és $x$ által kifeszített intervallum, $f \in C^{n+1}[a;b]$. Ekkor $\exists\, \xi_x \in [a;b]$, melyre:
> $$f(x) - p_n(x) = \frac{f^{(n+1)}(\xi_x)}{(n+1)!} \cdot \omega_n(x),$$
> ahol $\omega_n(x) = \prod_{i=0}^{n}(x - x_i)$.
>
> **Hibabecslés:**
> $$|f(x) - p_n(x)| \le \frac{M_{n+1}}{(n+1)!} \cdot |\omega_n(x)|, \quad M_{n+1} = \max_{\xi\in[a,b]}|f^{(n+1)}(\xi)|.$$

In [ ]:
# Hibaformula szemléltetése: sin(x), különböző n értékekre
f     = np.sin
a, b  = -np.pi/2, np.pi/2
t     = np.linspace(a, b, 500)

fig, axes = plt.subplots(1, 3, figsize=(13, 4))

for idx, n in enumerate([1, 2, 4]):
    xs = np.linspace(a, b, n+1)
    ys = f(xs)
    p_vals = np.array([lagrange_interp(xs, ys, ti) for ti in t])
    err    = np.abs(f(t) - p_vals)
    
    # hibakorlát: M_{n+1}/(n+1)! * max|omega|
    M = 1.0  # |sin^(n+1)| <= 1
    omega = np.array([np.prod(np.abs(ti - xs)) for ti in t])
    bound = M / np.math.factorial(n+1) * omega
    
    ax = axes[idx]
    ax.semilogy(t, err,   'b-',  lw=2,  label='tényleges hiba')
    ax.semilogy(t, bound, 'r--', lw=1.5, label='hibakorlát')
    ax.set_title(f'$n={n}$, alappontok: {n+1} db')
    ax.set_xlabel('$x$')
    if idx == 0: ax.set_ylabel('$|f(x) - p_n(x)|$')
    ax.legend(fontsize=8)

plt.suptitle('Hibaformula szemléltetése: $f(x) = \\sin(x)$', fontsize=12)
plt.tight_layout()
plt.show()

---
## 4. Newton-alak (osztott differenciák)

> **Definíció (osztott differenciák).**
> - 0-adrendű: $f[x_i] := f(x_i)$
> - Elsőrendű: $f[x_i, x_{i+1}] := \dfrac{f(x_{i+1}) - f(x_i)}{x_{i+1} - x_i}$
> - $k$-adrendű: $f[x_i, \ldots, x_{i+k}] := \dfrac{f[x_{i+1}, \ldots, x_{i+k}] - f[x_i, \ldots, x_{i+k-1}]}{x_{i+k} - x_i}$

> **Tétel (Newton-alak).**
> $$N_n(x) := f[x_0] + \sum_{k=1}^{n} f[x_0, \ldots, x_k] \cdot \omega_{k-1}(x) \equiv L_n(x),$$
> ahol $\omega_{k-1}(x) = (x-x_0)(x-x_1)\cdots(x-x_{k-1})$.
>
> **Rekurzív frissítés:** $N_{n+1}(x) = N_n(x) + f[x_0, \ldots, x_{n+1}] \cdot \omega_n(x).$

**Előny:** Ha új alappontot adunk hozzá, csak egy tagot kell hozzáírni — nem kell az egész interpolációt újraszámolni.

In [ ]:
def divided_diff_table(xs, ys):
    """Osztott differencia táblázat. T[i,k] = f[x_i,...,x_{i+k}]."""
    n = len(xs)
    T = np.zeros((n, n))
    T[:, 0] = np.array(ys, dtype=float)
    for k in range(1, n):
        for i in range(n - k):
            T[i, k] = (T[i+1, k-1] - T[i, k-1]) / (xs[i+k] - xs[i])
    return T

def newton_eval(xs, coeffs, t):
    """Newton-alak kiértékelése Horner-sémával."""
    n = len(coeffs)
    result = coeffs[n-1]
    for k in range(n-2, -1, -1):
        result = result * (t - xs[k]) + coeffs[k]
    return result

# Kidolgozott példa: f(x) = x³ - 2x + 1
xs = np.array([0., 1., 2., 3.])
ys = xs**3 - 2*xs + 1

T = divided_diff_table(xs, ys)
coeffs = T[0, :]   # főátló = Newton-együtthatók

print('Osztott differencia táblázat:')
print(f'{"x":>6}  {"f[xi]":>10}', end='')
for k in range(1, len(xs)):
    print(f'  {"f["+",".join(["x"+str(i) for i in range(k+1)])+"]": >14}', end='')
print()
print('-' * 60)
for i in range(len(xs)):
    row = f'{xs[i]:>6.1f}  {T[i,0]:>10.4f}'
    for k in range(1, len(xs)-i):
        row += f'  {T[i,k]:>14.4f}'
    print(row)

print(f'\nNewton-együtthatók: {coeffs}')

# Ellenőrzés
t_test = np.linspace(-0.5, 3.5, 300)
p_newton  = np.array([newton_eval(xs, coeffs, ti) for ti in t_test])
p_lagrange= np.array([lagrange_interp(xs, ys, ti) for ti in t_test])

plt.figure(figsize=(8, 4))
plt.plot(t_test, t_test**3 - 2*t_test + 1, 'b-', lw=2, label='$f(x)=x^3-2x+1$')
plt.plot(t_test, p_newton, 'r--', lw=2, label='Newton-alak $N_3$')
plt.scatter(xs, ys, s=80, color='k', zorder=5, label='alappontok')
plt.legend()
plt.title('Newton-alak: $f(x) = x^3 - 2x + 1$ (pontos illeszkedés)')
plt.show()
print(f'Max eltérés Newton–Lagrange: {np.max(np.abs(p_newton - p_lagrange)):.2e}')

### Osztott differenciák tulajdonságai

> **Tétel.**
> 1. $f[x_0, \ldots, x_k] = \sum_{j=0}^{k} \dfrac{f(x_j)}{\omega_k'(x_j)}$
> 2. $f[x_{\sigma(0)}, \ldots, x_{\sigma(k)}] = f[x_0, \ldots, x_k]$ (szimmetria bármely $\sigma$ permutációra)

**Következmény:** Az osztott differencia sorrendtől független — tetszőleges sorrendben vehetjük az alappontokat.

---
## 5. Hermite-interpoláció

> **Definíció.**  
> Adottak $x_0, \ldots, x_k \in [a;b]$ alappontok, $m_0, \ldots, m_k \in \mathbb{N}$ multiplicitások és a megfelelő derivált értékek $y_i^{(j)} = f^{(j)}(x_i)$. Legyen $m := \sum_{i=0}^{k} m_i - 1$. Olyan $H_m \in P_m$ polinomot keresünk, melyre
> $$H_m^{(j)}(x_i) = y_i^{(j)}, \quad (i=0,\ldots,k;\ j=0,\ldots,m_i-1).$$

**Speciális esetek:**
- $\forall m_i = 1$: Lagrange-interpoláció
- $\forall m_i = 2$: Fejér–Hermite-interpoláció ($f$ és $f'$ adott minden pontban)
- $k=0, m_0=m+1$: Taylor-polinom

**Newton-alak Hermite esetén:** Azonos alappontoknál az osztott differencia határátmenettel számítható:
$$f[\underbrace{x_i, x_i, \ldots, x_i}_{k+1}] := \frac{f^{(k)}(x_i)}{k!}$$
Ezért a Hermite-polinomot is Newton-alakban írhatjuk fel: minden alappontot annyiszor veszünk fel, amennyi a multiplicitása.

In [ ]:
def hermite_dd_table(xs_rep, ys_rep, dys_dict):
    """
    Hermite osztott differencia táblázat.
    xs_rep: alappontok ismétléssel (pl. [0,0,1,1,2])
    ys_rep: f értékek ismétléssel
    dys_dict: {x: [f(x), f'(x), f''(x)/2!, ...]} derivált értékek
    """
    n = len(xs_rep)
    T = np.zeros((n, n))
    T[:, 0] = ys_rep
    for k in range(1, n):
        for i in range(n - k):
            if xs_rep[i] == xs_rep[i+k]:  # azonos alappontok: derivált
                T[i, k] = dys_dict[xs_rep[i]][k]
            else:
                T[i, k] = (T[i+1, k-1] - T[i, k-1]) / (xs_rep[i+k] - xs_rep[i])
    return T

# Példa: f(x) = sin(x), m_0=m_1=2 (Fejér-Hermite, két pont)
# Alappontok: x0=0, x1=pi/2; f(0)=0, f'(0)=1, f(pi/2)=1, f'(pi/2)=0
x0, x1 = 0.0, np.pi/2
xs_rep = [x0, x0, x1, x1]
ys_rep = [np.sin(x0), np.sin(x0), np.sin(x1), np.sin(x1)]
dys_dict = {
    x0: [np.sin(x0), np.cos(x0), -np.sin(x0)/2],  # f, f', f''/2!
    x1: [np.sin(x1), np.cos(x1), -np.sin(x1)/2],
}

T = hermite_dd_table(xs_rep, ys_rep, dys_dict)
coeffs_H = T[0, :]

print('Hermite osztott differencia táblázat (m0=m1=2):')
print(f'{"x":>8}  {"f[xi]": >10}  {"1.od": >10}  {"2.od": >10}  {"3.od": >10}')
for i in range(4):
    row = f'{xs_rep[i]:>8.4f}  {T[i,0]:>10.6f}'
    for k in range(1, 4-i):
        row += f'  {T[i,k]:>10.6f}'
    print(row)

# Kiértékelés
t = np.linspace(-0.2, np.pi/2 + 0.2, 300)
H_vals = np.array([newton_eval(xs_rep, coeffs_H, ti) for ti in t])

plt.figure(figsize=(9, 4))
plt.plot(t, np.sin(t), 'b-', lw=2, label='$\\sin(x)$')
plt.plot(t, H_vals, 'r--', lw=1.8, label='$H_3(x)$ (Fejér-Hermite, $n=1$)')

# Lagrange összehasonlítás
xs_L = np.array([x0, x1])
ys_L = np.sin(xs_L)
L_vals = np.array([lagrange_interp(xs_L, ys_L, ti) for ti in t])
plt.plot(t, L_vals, 'g:', lw=1.5, label='$L_1(x)$ (Lagrange, $n=1$)')

plt.scatter([x0, x1], [np.sin(x0), np.sin(x1)], s=80, color='k', zorder=5)
plt.legend()
plt.title('Hermite vs Lagrange: $\\sin(x)$ közelítése 2 alapponton')
plt.show()

print(f'Max hiba – Hermite: {np.max(np.abs(np.sin(t)-H_vals)):.2e}')
print(f'Max hiba – Lagrange:{np.max(np.abs(np.sin(t)-L_vals)):.2e}')

### Hermite hibaformula

> **Tétel.**  
> Ha $f \in C^{m+1}[a;b]$, akkor
> $$f(x) - H_m(x) = \frac{f^{(m+1)}(\xi_x)}{(m+1)!} \cdot \Omega_m(x),$$
> ahol $\Omega_m(x) = \prod_{i=0}^{k}(x-x_i)^{m_i}$.

---
## 6. Csebisev-polinomok és optimális alappontok

> **Definíció.**  $T_n(x) := \cos(n \cdot \arccos(x))$, $x \in [-1;1]$ — az $n$-edfokú (elsőfajú) **Csebisev-polinom**.

> **Rekurzió (1. tétel):**
> $T_0(x) = 1,\ T_1(x) = x,\quad T_{n+1}(x) = 2x\,T_n(x) - T_{n-1}(x).$

> **Csebisev-tétel (4. tétel – extremális tulajdonság):**
> $$\min_{Q \in P_n^{(1)}} \|Q\|_\infty = \|\tilde{T}_n\|_\infty = \frac{1}{2^{n-1}},$$
> ahol $\tilde{T}_n = \frac{1}{2^{n-1}} T_n$ az 1 főegyütthatós Csebisev-polinom.

**Alkalmazás a hibaformulában:** Az $\omega_n(x) = \prod_{i=0}^{n}(x-x_i)$ tényező minimális, ha az alappontok a Csebisev-gyökök:
$$x_k = \cos\!\left(\frac{2k+1}{2(n+1)}\pi\right), \quad k=0,\ldots,n.$$

**Az interpoláció hibája $[-1;1]$-en Csebisev-alappontokkal:**
$$\|f - L_n\|_\infty \le \frac{M_{n+1}}{(n+1)!} \cdot \frac{1}{2^n}.$$

In [ ]:
def chebyshev_nodes(n, a=-1, b=1):
    """n+1 db Csebisev-gyök [a,b]-n."""
    ks = np.arange(n+1)
    nodes_11 = np.cos((2*ks + 1) / (2*(n+1)) * np.pi)
    return 0.5*(b-a)*nodes_11 + 0.5*(a+b)

def chebyshev_poly(n, x):
    """T_n(x) rekurzióval."""
    if n == 0: return np.ones_like(x, dtype=float)
    if n == 1: return np.array(x, dtype=float)
    T_prev, T_curr = np.ones_like(x, dtype=float), np.array(x, dtype=float)
    for _ in range(2, n+1):
        T_prev, T_curr = T_curr, 2*x*T_curr - T_prev
    return T_curr

# Runge-jelenség: Csebisev vs equidistant alappontok
f_runge = lambda x: 1 / (1 + 25*x**2)   # Runge-függvény
a, b    = -1, 1
t       = np.linspace(a, b, 500)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
n_values  = [5, 10, 15]
colors    = ['tab:orange', 'tab:green', 'tab:red']

for ax, title, node_fn in [
    (axes[0], 'Equidistans alappontok',  lambda n: np.linspace(a, b, n+1)),
    (axes[1], 'Csebisev-alappontok',     lambda n: chebyshev_nodes(n, a, b))
]:
    ax.plot(t, f_runge(t), 'b-', lw=2.5, label='$f(x) = 1/(1+25x^2)$', zorder=3)
    for n, col in zip(n_values, colors):
        xs = node_fn(n)
        ys = f_runge(xs)
        p_vals = np.array([lagrange_interp(xs, ys, ti) for ti in t])
        ax.plot(t, p_vals, '--', color=col, lw=1.5, label=f'$n={n}$')
    ax.set_ylim(-0.4, 1.3)
    ax.set_title(title)
    ax.legend(fontsize=9)

plt.suptitle('Runge-jelenség: equidistans vs Csebisev alappontok', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Csebisev-polinomok és gyökeik vizualizálása
x = np.linspace(-1, 1, 500)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

for n in [1, 2, 3, 4, 8]:
    ax1.plot(x, chebyshev_poly(n, x), lw=1.5, label=f'$T_{n}$')
ax1.axhline(0, color='k', lw=0.7)
ax1.axhline(1, color='k', lw=0.5, ls=':')
ax1.axhline(-1, color='k', lw=0.5, ls=':')
ax1.set_ylim(-1.3, 1.3)
ax1.set_title('Csebisev-polinomok $T_n(x)$')
ax1.legend(fontsize=8, ncol=2)

# Gyökök eloszlása
for n in [5, 10, 15]:
    nodes = chebyshev_nodes(n-1)
    ax2.scatter(nodes, np.full_like(nodes, n), s=20, label=f'$n={n}$')
ax2.set_yticks([5, 10, 15])
ax2.set_title('Csebisev-gyökök sűrűsödése a széleken')
ax2.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Hibacsökkenés: equidistans vs Csebisev, n növelésével
ns = range(2, 20)
err_eq  = []
err_cheb = []

for n in ns:
    # Equidistans
    xs = np.linspace(a, b, n+1)
    ys = f_runge(xs)
    p_eq = np.array([lagrange_interp(xs, ys, ti) for ti in t])
    err_eq.append(np.max(np.abs(f_runge(t) - p_eq)))
    
    # Csebisev
    xs_c = chebyshev_nodes(n)
    ys_c = f_runge(xs_c)
    p_ch = np.array([lagrange_interp(xs_c, ys_c, ti) for ti in t])
    err_cheb.append(np.max(np.abs(f_runge(t) - p_ch)))

plt.figure(figsize=(8, 4))
plt.semilogy(list(ns), err_eq,   'r-o', markersize=5, label='Equidistans')
plt.semilogy(list(ns), err_cheb, 'b-o', markersize=5, label='Csebisev')
plt.xlabel('$n$ (polinom foka)')
plt.ylabel('$\\|f - L_n\\|_\\infty$')
plt.title('Max hiba $n$ függvényében: Runge-függvény')
plt.legend()
plt.show()

---
## 7. Inverz interpoláció

**Feladat:** Ha $y_0 = f(x^*)$ adott, keressük $x^*$-t. Ötlet: interpoláljuk az $(y_i, x_i)$ párokat (azaz cseréljük fel a szerepüket).

**Előfeltétel:** $f$ szigorúan monoton legyen az alappontok körül.

In [ ]:
# Inverz interpoláció példa: keressük x-t, ahol sin(x) = 0.6
y_target = 0.6
x_true   = np.arcsin(y_target)   # pontos megoldás

# Alappontok: x értékek, melyekre sin(x) ismert
xs = np.array([0.0, np.pi/6, np.pi/4, np.pi/3, np.pi/2])
ys = np.sin(xs)

# Inverz: (y_i, x_i) párokat interpoláljuk
x_inv = lagrange_interp(ys, xs, y_target)

print(f'Keresett y = {y_target}')
print(f'Pontos x*  = arcsin({y_target}) = {x_true:.8f}')
print(f'Inv. interp = {x_inv:.8f}')
print(f'Hiba       = {abs(x_inv - x_true):.2e}')

# Vizualizáció
t = np.linspace(0, np.pi/2, 300)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

ax1.plot(t, np.sin(t), 'b-', lw=2, label='$\\sin(x)$')
ax1.scatter(xs, ys, s=60, color='k', zorder=5, label='alappontok')
ax1.axhline(y_target, color='r', ls='--', lw=1.2, label=f'$y={y_target}$')
ax1.axvline(x_true, color='g', ls=':', lw=1.5, label=f'$x^*={x_true:.4f}$')
ax1.legend(fontsize=9)
ax1.set_title('sin(x) és a keresett pont')

# Inverz polinom ábrája
y_range = np.linspace(min(ys)-0.05, max(ys)+0.05, 300)
p_inv   = np.array([lagrange_interp(ys, xs, yi) for yi in y_range])
ax2.plot(y_range, p_inv, 'r-', lw=2, label='inverz interp. polinom')
ax2.scatter(ys, xs, s=60, color='k', zorder=5)
ax2.axvline(y_target, color='b', ls='--', lw=1.2)
ax2.axhline(x_inv, color='g', ls=':', lw=1.5, label=f'$x_{{inv}}={x_inv:.4f}$')
ax2.set_xlabel('y'); ax2.set_ylabel('x')
ax2.legend(fontsize=9)
ax2.set_title('Inverz interpoláció')

plt.tight_layout()
plt.show()

---
## Összefoglalás

| Módszer | Polinom foka | Előny | Megjegyzés |
|---------|-------------|-------|------------|
| Lagrange-alak | $n$ | szemléletes, egyszerű | $n+1$ alapponthoz |
| Newton-alak | $n$ | rekurzívan bővíthető | Horner-séma hatékony |
| Hermite-interpoláció | $m = \sum m_i - 1$ | derivált értékeket is illeszti | Taylor mint speciális eset |
| Csebisev-alappontok | $n$ | minimális $\|\omega_n\|_\infty$ | elkerüli Runge-jelenséget |

**Konvergencia:**
- **Faber-tétel:** minden alappontrendszerhez $\exists f \in C[a;b]$, hogy $L_n \not\to f$.
- **Marcinkiewicz-tétel:** minden $f \in C[a;b]$-hez $\exists$ alappontrendszer, hogy $L_n \to f$.
- Ha $f \in C^\infty[a;b]$ és $\|f^{(n)}\|_\infty \le M^n$, akkor bármely alappontnál $L_n \to f$.